In [2]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models


In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/Shareddrives/ECI271_Image_Data/data"

if os.path.exists(DATA_ROOT):
    print(os.listdir(DATA_ROOT)[:20])
else:
    print("path not found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['families.txt', 'variants.txt', 'manufacturers.txt', 'images_variant_trainval.txt', 'images_box.txt', 'images_variant_train.txt', 'images_variant_val.txt', 'images_val.txt', 'images_train.txt', 'images_test.txt', 'images_family_train.txt', 'images_family_test.txt', 'images_family_trainval.txt', 'images_family_val.txt', 'images_variant_test.txt', 'images_manufacturer_train.txt', 'images_manufacturer_trainval.txt', 'images_manufacturer_test.txt', 'images_manufacturer_val.txt', 'images']


In [4]:
class AircraftDataset(Dataset):

    def __init__(
        self,
        root,
        split,
        label_level="variant",
        transform=None,
        class_to_idx=None
    ):
        self.root = root
        self.split = split
        self.label_level = label_level
        self.transform = transform

        self.image_dir = os.path.join(root, "images")

        label_file = os.path.join(root, f"images_{label_level}_{split}.txt")
        box_file = os.path.join(root, "images_box.txt")

        self.boxes = self._load_boxes(box_file)
        self.samples = self._load_labels(label_file)

        if class_to_idx is None:
            classes = sorted(set(label for _, label in self.samples))
            self.class_to_idx = {c: i for i, c in enumerate(classes)}
        else:
            self.class_to_idx = class_to_idx

        self.classes = list(self.class_to_idx.keys())

    def _load_boxes(self, path):
        boxes = {}

        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()

                img_id = parts[0]
                xmin, ymin, xmax, ymax = map(float, parts[1:])

                boxes[img_id] = (xmin, ymin, xmax, ymax)

        return boxes

    def _load_labels(self, path):
        samples = []

        with open(path, "r") as f:
            for line in f:
                parts = line.strip().split()

                img_id = parts[0]
                label = " ".join(parts[1:])

                samples.append((img_id, label))

        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_id, label_name = self.samples[idx]

        img_path = os.path.join(self.image_dir, img_id + ".jpg")
        img = Image.open(img_path).convert("RGB")

        w, h = img.size
        img = img.crop((0, 0, w, h - 20))
        if img_id in self.boxes:
            xmin, ymin, xmax, ymax = self.boxes[img_id]

            xmin = max(0, int(xmin) - 1)
            ymin = max(0, int(ymin) - 1)

            xmax = min(w, int(xmax))
            ymax = min(h - 20, int(ymax))

            if xmax > xmin and ymax > ymin:
                img = img.crop((xmin, ymin, xmax, ymax))

        if self.transform:
            img = self.transform(img)

        label = self.class_to_idx[label_name]

        return img, label

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LR = 3e-5
WEIGHT_DECAY = 1e-4

LABEL_LEVEL = "variant"

SAVE_PATH = "/content/drive/MyDrive/best_aircraft_vit_base.pth"
LATEST_PATH = "/content/drive/MyDrive/latest_aircraft_vit_base_checkpoint.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#data augmentation
train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.65, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),

    # blur
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.30),

    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.4,
            saturation=0.3,
            hue=0.03
        )
    ], p=0.60),

    # lower res
    transforms.RandomApply([
        transforms.Resize((80, 80)),
        transforms.Resize((IMG_SIZE, IMG_SIZE))
    ], p=0.30),

    transforms.ToTensor(),

    # randon sensor noise
    transforms.RandomApply([
        transforms.Lambda(lambda x: torch.clamp(x + 0.05 * torch.randn_like(x), 0, 1))
    ], p=0.40),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
train_ds = AircraftDataset(
    root=DATA_ROOT,
    split="train",
    label_level=LABEL_LEVEL,
    transform=train_tfms
)

val_ds = AircraftDataset(
    root=DATA_ROOT,
    split="val",
    label_level=LABEL_LEVEL,
    transform=eval_tfms,
    class_to_idx=train_ds.class_to_idx
)

test_ds = AircraftDataset(
    root=DATA_ROOT,
    split="test",
    label_level=LABEL_LEVEL,
    transform=eval_tfms,
    class_to_idx=train_ds.class_to_idx
)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))
print("Number of classes:", len(train_ds.classes))
print("First 10 classes:", train_ds.classes[:10])

Train: 3334
Val: 3333
Test: 3333
Number of classes: 100
First 10 classes: ['707-320', '727-200', '737-200', '737-300', '737-400', '737-500', '737-600', '737-700', '737-800', '737-900']


In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
weights = models.ViT_B_16_Weights.IMAGENET1K_V1

model = models.vit_b_16(weights=weights)

num_classes = len(train_ds.classes)

model.heads.head = nn.Linear(
    model.heads.head.in_features,
    num_classes
)

model = model.to(device)

print("Number of classes:", num_classes)
print(model.heads)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 128MB/s]


Number of classes: 100
Sequential(
  (head): Linear(in_features=768, out_features=100, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.3
)

In [ ]:
def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            total_correct += (preds == labels).sum().item()

            total_samples += labels.size(0)

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples

    return avg_loss, avg_acc

In [ ]:
best_val_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())
start_epoch = 0

for epoch in range(start_epoch, EPOCHS):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    start_time = time.time()

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, preds = torch.max(outputs, 1)
        running_correct += (preds == labels).sum().item()

        total_samples += labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_loader)}")

    scheduler.step()

    train_loss = running_loss / total_samples
    train_acc = running_correct / total_samples

    val_loss, val_acc = evaluate(model, val_loader)

    elapsed = time.time() - start_time

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    print(f"Time: {elapsed/60:.2f} minutes")

    torch.save(
        {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "best_val_acc": best_val_acc,
            "class_names": train_ds.classes,
            "class_to_idx": train_ds.class_to_idx
        },
        LATEST_PATH
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_weights = copy.deepcopy(model.state_dict())

        torch.save(
            {
                "epoch": epoch,
                "model_state": best_weights,
                "best_val_acc": best_val_acc,
                "class_names": train_ds.classes,
                "class_to_idx": train_ds.class_to_idx
            },
            SAVE_PATH
        )

        print("Saved new best model")

    print("-" * 60)

model.load_state_dict(best_weights)
print("Best validation accuracy:", best_val_acc)

Epoch 1, Batch 50/209
Epoch 1, Batch 100/209
Epoch 1, Batch 150/209
Epoch 1, Batch 200/209
Epoch 1/15
Train Loss: 4.1515 | Train Acc: 0.1047
Val Loss:   3.3321 | Val Acc:   0.2214
Time: 45.74 minutes
Saved new best model
------------------------------------------------------------
Epoch 2, Batch 50/209
Epoch 2, Batch 100/209
Epoch 2, Batch 150/209
Epoch 2, Batch 200/209
Epoch 2/15
Train Loss: 2.6658 | Train Acc: 0.4211
Val Loss:   2.1806 | Val Acc:   0.5272
Time: 3.55 minutes
Saved new best model
------------------------------------------------------------
Epoch 3, Batch 50/209
Epoch 3, Batch 100/209
Epoch 3, Batch 150/209
Epoch 3, Batch 200/209
Epoch 3/15
Train Loss: 1.7313 | Train Acc: 0.6431
Val Loss:   1.5583 | Val Acc:   0.6214
Time: 3.96 minutes
Saved new best model
------------------------------------------------------------
Epoch 4, Batch 50/209
Epoch 4, Batch 100/209
Epoch 4, Batch 150/209
Epoch 4, Batch 200/209
Epoch 4/15
Train Loss: 1.1631 | Train Acc: 0.7660
Val Loss:   1.3

In [ ]:
# for checkpoints if failed
"""checkpoint = torch.load(LATEST_PATH, map_location=device)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
scheduler.load_state_dict(checkpoint["scheduler_state"])

start_epoch = checkpoint["epoch"] + 1
best_val_acc = checkpoint["best_val_acc"]

print("Resumed from epoch:", start_epoch)
print("Best val acc so far:", best_val_acc)"""

'checkpoint = torch.load(LATEST_PATH, map_location=device)\n\nmodel.load_state_dict(checkpoint["model_state"])\noptimizer.load_state_dict(checkpoint["optimizer_state"])\nscheduler.load_state_dict(checkpoint["scheduler_state"])\n\nstart_epoch = checkpoint["epoch"] + 1\nbest_val_acc = checkpoint["best_val_acc"]\n\nprint("Resumed from epoch:", start_epoch)\nprint("Best val acc so far:", best_val_acc)'

In [ ]:
checkpoint = torch.load(SAVE_PATH, map_location=device)

class_names = checkpoint["class_names"]
num_classes = len(class_names)

model = models.vit_b_16(weights=None)

model.heads.head = nn.Linear(
    model.heads.head.in_features,
    num_classes
)

model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

print("Loaded best ViT-Base model")
print("Best validation accuracy:", checkpoint.get("best_val_acc", "not saved"))

Loaded best ViT-Base model
Best validation accuracy: 0.7977797779777978


In [ ]:
test_loss, test_acc = evaluate(model, test_loader)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

Test Loss: 0.7550578737559348
Test Accuracy: 0.8052805280528053
